### Open AI Agents SDK — Functions and Tools

"The OpenAI Agents SDK is a lightweight framework for building AI agents."

This notebook extends the intro workflow by wrapping Python functions as **tools** with `@function_tool`, passing them to `Agent(tools=[...])`, and letting the model call them during `Runner.run`.

Documentation:
https://openai.github.io/openai-agents-python/

Tools documentation:
https://openai.github.io/openai-agents-python/tools/

Github Repo:
https://github.com/openai/openai-agents-python

In [2]:
%pip install -q openai-agents python-dotenv wikipedia

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: C:\Users\dbenn\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import json
import os

from dotenv import load_dotenv
from IPython.display import Markdown, display

from agents import Agent, Runner, function_tool, trace

##### Packages Overview

**openai-agents**  
For creating and orchestrating agents with function tools  
Classes:
  - Agent, Runner, trace, function_tool (decorator), FunctionTool

**python-dotenv**  
Loads environment variables from a `.env` file (e.g. `OPENAI_API_KEY`)

**wikipedia**  
Python wrapper for the Wikipedia API — used by our `wikipedia_search` tool

In [3]:
load_dotenv()

print("OpenAI API key loaded:", os.getenv("OPENAI_API_KEY") is not None)

OpenAI API key loaded: True


- https://aistudio.google.com/app/
- https://platform.openai.com/login

## Define Function Tools

Use `@function_tool` to wrap a Python function so the agent can call it. The SDK reads each function's **name**, **docstring**, and **type hints** to build the tool schema the model sees.

The tool below calls **live Wikipedia** via the [`wikipedia`](https://pypi.org/project/wikipedia/) package (requires internet).

In [4]:
import wikipedia

wikipedia.set_lang("en")


@function_tool
def wikipedia_search(query: str, max_results: int = 3) -> str:
    """Search Wikipedia and return short summaries for the top articles."""
    try:
        titles = wikipedia.search(query, results=max_results)
        if not titles:
            return f"No results for '{query}'."

        parts = []
        for title in titles:
            summary = wikipedia.summary(title, sentences=5, auto_suggest=True)
            page = wikipedia.page(title, auto_suggest=True)
            parts.append(f"- **{page.title}** ({page.url})\n  {summary}")
        return "\n".join(parts)
    except Exception as e:
        return f"Wikipedia error: {e}"

## Simple Example with Tools

One agent with a single tool — the model decides when to call `wikipedia_search` based on the user's question.

In [6]:
fact_finder = Agent(
    name="Fact Finder",
    instructions=(
        "You are a concise research assistant. "
        "Always call wikipedia_search before answering factual questions. "
        "Summarize the tool output in 3–5 bullets and include Wikipedia URLs."
    ),
    model="gpt-4o-mini",
    tools=[wikipedia_search],
)

result = await Runner.run(fact_finder, "Tell me something interesting about the FIFA world cup?")
display(Markdown(result.final_output))

Here are some interesting facts about the FIFA World Cup:

- **History and Inception**: The first FIFA World Cup was held in 1930, initiated by FIFA president Jules Rimet. The trophy was named the Jules Rimet Cup in his honor in 1946 ([History of the FIFA World Cup](https://en.wikipedia.org/wiki/History_of_the_FIFA_World_Cup)).
  
- **Tournament Frequency**: The World Cup is held every four years, with the exceptions being 1942 and 1946 due to World War II ([List of FIFA World Cup finals](https://en.wikipedia.org/wiki/List_of_FIFA_World_Cup_finals)).
  
- **Expansion Over Time**: The tournament began with 13 teams, and has grown in size and format, currently featuring 48 teams in the final tournament after a rigorous qualifying process involving over 200 teams ([History of the FIFA World Cup](https://en.wikipedia.org/wiki/History_of_the_FIFA_World_Cup)).
  
- **Winning Nations**: Brazil is the most successful team, with five championships, while Argentina is the current champion as of the 2022 World Cup ([FIFA World Cup records and statistics](https://en.wikipedia.org/wiki/FIFA_World_Cup_records_and_statistics)).
  
- **Notable Participation**: 80 national teams have competed in the tournament finals, with Brazil the only nation to participate in all 22 editions ([FIFA World Cup records and statistics](https://en.wikipedia.org/wiki/FIFA_World_Cup_records_and_statistics)).

### Agentic Workflow with Tools

**Overview**  
Build a multi-agent workflow using OpenAI Agents SDK where agents can call Python function tools during research. Same pattern as `agents_sdk_intro.ipynb`, but agents use live Wikipedia tools instead of relying only on model knowledge.

**Agents**
- **Researcher** – gathers facts and details using `wikipedia_search`
- **Reporter** – produces a professional report and can fact-check with `wikipedia_search`

**Tools**
- `wikipedia_search` — search a topic and return short summaries with URLs

### Create Agents

In [44]:
researcher_inst = "You are a skilled and resourceful researcher. Your job is to deeply investigate any assigned topic, intelligently leverage your knowledge and available tools, and synthesize relevant, credible information from trustworthy sources. Your research should emphasize both recent developments and core facts, highlight significance and context, and clearly cite your sources when possible. Focus on accuracy, clarity, and actionable insight in your findings."

reporter_inst = "You are a meticulous analyst renowned for your keen attention to detail. You excel at transforming complex information into clear, concise, and actionable reports, making even the most intricate data accessible and understandable for your audience."

In [45]:
researcher = Agent(
    name="Professional Researcher",
    instructions=researcher_inst,
    model="gpt-4o-mini",
    tools=[wikipedia_search],
)

In [ ]:
reporter = Agent(
    name="Professional Reporter",
    instructions=reporter_inst,
    model="gpt-4.1-mini",
    tools=[wikipedia_search]
)

#### Do Initial Research

In [47]:
topic = "ancient egypt"

In [48]:
with trace("research with function tools"):
    result = await Runner.run(
        researcher,
        f"Research the topic: {topic}. "
        "Use your tools find interesting facts, people, dates, events, sources, etc..."
        "Output 8–10 detailed markdown bullets plus a Sources section. "
        "Only return markdown (no enclosing triple backticks).",
    )
    research_result = result.final_output

In [49]:
display(Markdown(research_result))

- **Unification of Upper and Lower Egypt (circa 3150 BC)**: Ancient Egypt emerged around 3150 BC when King Menes, often identified with Narmer, united Upper and Lower Egypt. This event marked the foundation of one of the world's earliest and most influential civilizations, leading to the establishment of a centralized state along the Nile River. [Source](https://en.wikipedia.org/wiki/Ancient_Egypt).

- **Periods of Egyptian History**: The history of ancient Egypt is divided into three main periods: the Old Kingdom (circa 2686–2181 BC) known for its monumental architecture like the Pyramids; the Middle Kingdom (circa 2055–1650 BC) recognized for its literature and art; and the New Kingdom (circa 1550–1070 BC), the height of Egyptian power and territorial expansion into Nubia and the Levant. [Source](https://en.wikipedia.org/wiki/Ancient_Egypt).

- **Pyramids of Giza**: The Great Pyramid of Giza, built around 2580–2560 BC, is one of the Seven Wonders of the Ancient World and is a testament to the architectural ingenuity of the Egyptians. It was originally 146.6 meters tall and was built as a tomb for Pharaoh Khufu. [Source](https://en.wikipedia.org/wiki/Ancient_Egypt).

- **Ancient Egyptian Religion**: Polytheism was central to Egyptian culture, with about 1,500 deities worshipped. Major gods included Ra (the sun god), Osiris (god of the afterlife), and Isis (goddess of magic). Temples served as the centers of worship and were believed to be the homes of the gods, managed by the pharaohs, considered divine rulers. [Source](https://en.wikipedia.org/wiki/Ancient_Egyptian_religion).

- **Funerary Practices**: The ancient Egyptians believed in an afterlife and thus developed elaborate funerary practices, including mummification, which aimed to preserve the body for eternity. Rituals were performed to ensure safe passage to the afterlife, and tombs were filled with goods for use beyond death. [Source](https://en.wikipedia.org/wiki/Ancient_Egyptian_funerary_practices).

- **Agriculture and the Nile**: The civilization thrived due to the Nile River's seasonal flooding, which provided fertile soil for agriculture. The Egyptians innovated with basin irrigation, enabling them to cultivate wheat, barley, and flax, laying the agricultural foundation for their society. [Source](https://en.wikipedia.org/wiki/Ancient_Egyptian_agriculture).

- **Art and Aesthetics**: Ancient Egyptian art was characterized by its adherence to convention and strict adherence to rules. Art was used to serve functional purposes in tombs and temples, portraying the ideals of beauty and representing the gods and deceased. Notably, the ancient Egyptians had no word for "art." [Source](https://en.wikipedia.org/wiki/Art_of_ancient_Egypt).

- **Famous Figures**: Prominent individuals include Pharaoh Tutankhamun, known for his richly furnished tomb discovered in 1922; Queen Hatshepsut, one of the few female pharaohs who ruled during the New Kingdom; and Ramses II, known for his military leadership and monumental temple constructions. [Source](https://en.wikipedia.org/wiki/Famous_people_in_ancient_Egypt).

- **Legacy of Ancient Egypt**: The cultural contributions of ancient Egypt have had lasting impacts on art, architecture, writing, and religion. Many aspects of their civilization influenced neighboring cultures and continue to be subjects of study and fascination in modern times. [Source](https://en.wikipedia.org/wiki/Ancient_Egypt).

### Sources
- [Ancient Egypt - Wikipedia](https://en.wikipedia.org/wiki/Ancient_Egypt)
- [Ancient Egyptian Religion - Wikipedia](https://en.wikipedia.org/wiki/Ancient_Egyptian_religion)
- [Art of Ancient Egypt - Wikipedia](https://en.wikipedia.org/wiki/Art_of_ancient_Egypt)
- [Ancient Egyptian Funerary Practices - Wikipedia](https://en.wikipedia.org/wiki/Ancient_Egyptian_funerary_practices)
- [Ancient Egyptian Agriculture - Wikipedia](https://en.wikipedia.org/wiki/Ancient_Egyptian_agriculture)

#### Build the Report

In [50]:
with trace("building the report"):
    result = await Runner.run(
        reporter,
        f"You are provided with the following research notes: "
        f"{research_result} "
        "For each bullet point, expand it into a clear and comprehensive report section. "
        "Retain factual accuracy from the original notes and preserve any attributions to sources. "
        "Where appropriate, enrich each section with relevant supporting details. "
        "Your output should be a full report structured by main topics. "
        "Use your tools to fact check infromation provided in the research notes. "
        'At the end, include a concise "Sources" list. '
        "Format the output as Markdown (but omit enclosing triple backticks).",
    )
    report_result = result.final_output

In [51]:
display(Markdown(report_result))

# The Civilization of Ancient Egypt: A Comprehensive Overview

Ancient Egypt, emerging around 3150 BC with the unification of Upper and Lower Egypt, stands as one of the most influential civilizations in human history. This report presents a detailed examination of various facets of ancient Egyptian civilization, from its political foundations to its cultural legacies.

## Unification of Upper and Lower Egypt (circa 3150 BC)

Ancient Egypt's history pivots around the unification of Upper and Lower Egypt, traditionally attributed to King Menes, often identified with Narmer. This event marked the inception of a centralized state along the Nile River, fostering a rich cultural and political milieu. The duality of Upper and Lower Egypt reflects the ancient Egyptian worldview, a theme evident in their art and politics. Pharaohs were commonly referred to with titles exemplifying this unity, such as "Uniter of the Two Lands" (sematawy). This momentous event laid the groundwork for future dynasties and significant cultural achievements. [Source](https://en.wikipedia.org/wiki/Upper_and_Lower_Egypt)

## Periods of Egyptian History

The history of ancient Egypt is subdivided into three prominent periods: the **Old Kingdom** (circa 2686–2181 BC), known for monumental achievements such as the Pyramids; the **Middle Kingdom** (circa 2055–1650 BC), which heralded advancements in literature and art; and the **New Kingdom** (circa 1550–1070 BC), recognized as the apex of Egyptian power characterized by territorial expansions and military conquests. Each period contributed uniquely to the cultural and political tapestry of Egypt, shaping the civilization that would influence numerous neighboring societies. [Source](https://en.wikipedia.org/wiki/Ancient_Egypt)

## Pyramids of Giza

The Great Pyramid of Giza, constructed around 2580–2560 BC for Pharaoh Khufu, remains one of the Seven Wonders of the Ancient World. Originally standing at 146.6 meters, it was the tallest man-made structure for over 3,700 years. This architectural marvel underscores the Egyptians' exceptional engineering skills and serves as a testament to their religious beliefs regarding the afterlife. It symbolizes the grandeur of the Old Kingdom and plays a pivotal role in our understanding of ancient Egyptian funerary practices. [Source](https://en.wikipedia.org/wiki/Great_Pyramid_of_Giza)

## Ancient Egyptian Religion

Religion permeated every facet of ancient Egyptian life, characterized by polytheism and reverence for approximately 1,500 deities. Major gods, including Ra (the sun god), Osiris (god of the afterlife), and Isis (goddess of magic), were central to the Egyptians' worldview. Temples functioned as sacred spaces for worship, reinforcing the divine status of the pharaohs, who were seen as intermediaries between the gods and the people. Rituals, prayers, and offerings formed the bedrock of their religious practices, significantly influencing their cultural evolution. [Source](https://en.wikipedia.org/wiki/Ancient_Egyptian_religion)

## Funerary Practices

The elaborate funerary practices of the ancient Egyptians underscore their belief in the afterlife. Mummification aimed to preserve the body for eternity, ensuring the deceased's safe passage to the afterlife. Burials were often accompanied by grave goods essential for the journey beyond death, reflecting profound respect for the deceased. Over time, while certain practices evolved, the importance of mummification and the rituals surrounding funerals remained a constant in their culture. [Source](https://en.wikipedia.org/wiki/Ancient_Egyptian_funerary_practices)

## Agriculture and the Nile

Agriculture served as the backbone of ancient Egyptian civilization, relying heavily on the seasonal flooding of the Nile River. This predictable flooding enriched the soil, allowing for the cultivation of staple crops such as wheat and barley. Egyptians pioneered basin irrigation techniques, facilitating large-scale agriculture that sustained their growing population and contributed to the empire's wealth. The agriculture innovations played a crucial role in enabling societal stability and expansion. [Source](https://en.wikipedia.org/wiki/Ancient_Egyptian_agriculture)

## Art and Aesthetics

Ancient Egyptian art is marked by its adherence to convention and purpose rather than individual expression. It served functional roles in depicting religious narratives and ideals of beauty while significantly adhering to established artistic norms over millennia. The absence of a specific term for "art" in ancient Egyptian language highlights this pragmatic application, where art was deeply woven into the fabric of their daily and spiritual lives, especially in tombs and temples. [Source](https://en.wikipedia.org/wiki/Art_of_ancient_Egypt)

## Famous Figures

Prominent figures from ancient Egyptian history include Pharaoh Tutankhamun, renowned for his lavishly furnished tomb discovered in 1922, Queen Hatshepsut, one of the few female rulers in ancient Egypt, and Ramses II, celebrated for his military prowess and monumental constructions. Each of these figures played significant roles in shaping Egypt’s cultural and political landscape, leaving enduring legacies that continue to fascinate scholars and the public alike. [Source](https://en.wikipedia.org/wiki/Famous_people_in_ancient_Egypt)

## Legacy of Ancient Egypt

The cultural contributions of ancient Egypt, particularly in art, architecture, writing, and religion, have had a lasting influence on subsequent civilizations and continue to capture the imaginations of people worldwide. The legacies of their societal structures, mythologies, and artistic expressions remain relevant subjects of study, offering invaluable insights into the complexities of human civilization and the enduring impact of ancient Egyptian culture. [Source](https://en.wikipedia.org/wiki/Ancient_Egypt)

---

## Sources
- [Ancient Egypt - Wikipedia](https://en.wikipedia.org/wiki/Ancient_Egypt)
- [Ancient Egyptian Religion - Wikipedia](https://en.wikipedia.org/wiki/Ancient_Egyptian_religion)
- [Art of Ancient Egypt - Wikipedia](https://en.wikipedia.org/wiki/Art_of_ancient_Egypt)
- [Ancient Egyptian Funerary Practices - Wikipedia](https://en.wikipedia.org/wiki/Ancient_Egyptian_funerary_practices)
- [Ancient Egyptian Agriculture - Wikipedia](https://en.wikipedia.org/wiki/Ancient_Egyptian_agriculture)
- [Upper and Lower Egypt - Wikipedia](https://en.wikipedia.org/wiki/Upper_and_Lower_Egypt)
- [Great Pyramid of Giza - Wikipedia](https://en.wikipedia.org/wiki/Great_Pyramid_of_Giza)

#### Checkout the Traces

Tool calls appear in traces alongside model turns.

https://platform.openai.com/logs?api=traces